In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

try:
    import talib
    print('TA-Lib imported successfully')
except ImportError:
    print('TA-Lib not installed. Using alternative indicators.')
    talib = None

import yfinance as yf

In [ ]:
import os

DATA_DIR = '../data/raw/'
TICKERS = ['AAPL', 'AMZN', 'GOOG', 'MSFT', 'TSLA', 'META', 'NVDA']

def load_stock_data(ticker, data_dir=DATA_DIR):
    """Load stock CSV or fall back to yfinance download."""
    path = os.path.join(data_dir, f'{ticker}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
        print(f"Loaded {ticker} from CSV: {df.shape}")
    else:
        print(f"{ticker} not found locally, downloading from yfinance...")
        df = yf.download(ticker, start='2020-01-01', end='2024-01-01', auto_adjust=False)
        df.to_csv(path)
    return df

# Load all tickers into a dict
stocks = {ticker: load_stock_data(ticker) for ticker in TICKERS}

# Preview one
print(stocks['AAPL'].head())

In [ ]:
def clean_stock_df(df, ticker):
    """Standardize columns, handle missing values, validate types."""
    # Flatten MultiIndex columns (yfinance sometimes returns them)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    required_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"[{ticker}] Missing columns: {missing}")

    # Ensure numeric types
    for col in required_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    before = len(df)
    df = df.dropna(subset=['Close', 'Adj Close'])
    after = len(df)
    if before != after:
        print(f"[{ticker}] Dropped {before - after} rows with null Close/Adj Close")

    df = df.sort_index()
    return df

# Clean all
for ticker in TICKERS:
    stocks[ticker] = clean_stock_df(stocks[ticker], ticker)

# Summary
for ticker, df in stocks.items():
    print(f"{ticker}: {df.shape[0]} rows | {df.index.min().date()} → {df.index.max().date()}")

In [ ]:
def add_moving_averages(df):
    """Add SMA and EMA columns for multiple windows."""
    close = df['Adj Close'].values.astype(float)

    if talib:
        df['SMA_20']  = talib.SMA(close, timeperiod=20)
        df['SMA_50']  = talib.SMA(close, timeperiod=50)
        df['SMA_200'] = talib.SMA(close, timeperiod=200)
        df['EMA_12'] = talib.EMA(close, timeperiod=12)
        df['EMA_26'] = talib.EMA(close, timeperiod=26)
        df['EMA_50'] = talib.EMA(close, timeperiod=50)
    else:
        df['SMA_20']  = pd.Series(close).rolling(window=20).mean().values
        df['SMA_50']  = pd.Series(close).rolling(window=50).mean().values
        df['SMA_200'] = pd.Series(close).rolling(window=200).mean().values
        df['EMA_12'] = pd.Series(close).ewm(span=12).mean().values
        df['EMA_26'] = pd.Series(close).ewm(span=26).mean().values
        df['EMA_50'] = pd.Series(close).ewm(span=50).mean().values

    return df

for ticker in TICKERS:
    stocks[ticker] = add_moving_averages(stocks[ticker])

print("Moving averages added.")
print(stocks['AAPL'][['Adj Close','SMA_20','SMA_50','EMA_12']].tail(5))

In [ ]:
def add_rsi(df, period=14):
    close = df['Adj Close'].values.astype(float)
    if talib:
        df['RSI'] = talib.RSI(close, timeperiod=period)
    else:
        delta = pd.Series(close).diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))
    return df

for ticker in TICKERS:
    stocks[ticker] = add_rsi(stocks[ticker])

print("RSI added.")
print(stocks['AAPL'][['Adj Close','RSI']].tail(5))

In [ ]:
def add_macd(df):
    close = df['Adj Close'].values.astype(float)
    if talib:
        macd, signal, hist = talib.MACD(close,
                                         fastperiod=12,
                                         slowperiod=26,
                                         signalperiod=9)
        df['MACD']        = macd
        df['MACD_Signal'] = signal
        df['MACD_Hist']   = hist
    else:
        ema_12 = pd.Series(close).ewm(span=12).mean()
        ema_26 = pd.Series(close).ewm(span=26).mean()
        df['MACD'] = (ema_12 - ema_26).values
        df['MACD_Signal'] = df['MACD'].ewm(span=9).mean().values
        df['MACD_Hist'] = (df['MACD'] - df['MACD_Signal']).values
    return df

for ticker in TICKERS:
    stocks[ticker] = add_macd(stocks[ticker])

print("MACD added.")
print(stocks['AAPL'][['MACD','MACD_Signal','MACD_Hist']].tail(5))

In [ ]:
def add_bollinger_bands(df, period=20, nbdevup=2, nbdevdn=2):
    close = df['Adj Close'].values.astype(float)
    if talib:
        upper, middle, lower = talib.BBANDS(close,
                                             timeperiod=period,
                                             nbdevup=nbdevup,
                                             nbdevdn=nbdevdn)
        df['BB_Upper']  = upper
        df['BB_Middle'] = middle
        df['BB_Lower']  = lower
    else:
        sma = pd.Series(close).rolling(window=period).mean()
        std = pd.Series(close).rolling(window=period).std()
        df['BB_Upper']  = (sma + nbdevup * std).values
        df['BB_Middle'] = sma.values
        df['BB_Lower']  = (sma - nbdevdn * std).values
    return df

for ticker in TICKERS:
    stocks[ticker] = add_bollinger_bands(stocks[ticker])

print("Bollinger Bands added.")

In [ ]:
def add_pynance_metrics(df):
    """Add daily returns and rolling volatility."""
    df['daily_return']      = df['Adj Close'].pct_change() * 100
    df['log_return']        = np.log(df['Adj Close'] / df['Adj Close'].shift(1))
    df['rolling_vol_20']    = df['daily_return'].rolling(window=20).std()
    df['rolling_vol_60']    = df['daily_return'].rolling(window=60).std()
    df['cumulative_return'] = (1 + df['daily_return'] / 100).cumprod() - 1
    return df

for ticker in TICKERS:
    stocks[ticker] = add_pynance_metrics(stocks[ticker])

print("Financial metrics added.")
print(stocks['AAPL'][['daily_return','rolling_vol_20','cumulative_return']].tail(5))

In [ ]:
summary_rows = []

for ticker, df in stocks.items():
    row = {
        'Ticker'          : ticker,
        'Start'           : df.index.min().date(),
        'End'             : df.index.max().date(),
        'Mean Daily Ret %': round(df['daily_return'].mean(), 4),
        'Std Daily Ret %' : round(df['daily_return'].std(), 4),
        'Max Drawdown %'  : round(df['daily_return'].min(), 4),
        'Annualized Vol'  : round(df['daily_return'].std() * np.sqrt(252), 2),
        'Total Return %'  : round(df['cumulative_return'].iloc[-1] * 100, 2),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
def plot_price_with_ma(ticker, df, start='2022-01-01'):
    df_plot = df[df.index >= start].copy()

    plt.figure(figsize=(16, 6))
    plt.plot(df_plot.index, df_plot['Adj Close'], label='Adj Close', linewidth=1.2, color='black')
    plt.plot(df_plot.index, df_plot['SMA_20'],    label='SMA 20',    linewidth=1,   color='steelblue',  linestyle='--')
    plt.plot(df_plot.index, df_plot['SMA_50'],    label='SMA 50',    linewidth=1,   color='orange',     linestyle='--')
    plt.plot(df_plot.index, df_plot['SMA_200'],   label='SMA 200',   linewidth=1.2, color='red',        linestyle='-.')
    plt.fill_between(df_plot.index, df_plot['BB_Upper'], df_plot['BB_Lower'],
                     alpha=0.08, color='gray', label='Bollinger Bands')

    plt.title(f'{ticker} — Price & Moving Averages')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(f'../data/raw/{ticker}_price_ma.png', dpi=150)
    plt.show()

# Plot for each ticker
for ticker in TICKERS:
    plot_price_with_ma(ticker, stocks[ticker])

In [ ]:
def plot_rsi(ticker, df, start='2022-01-01'):
    df_plot = df[df.index >= start].copy()

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                              gridspec_kw={'height_ratios': [3, 1]})

    # Price
    axes[0].plot(df_plot.index, df_plot['Adj Close'], color='black', linewidth=1.2)
    axes[0].set_title(f'{ticker} — Price & RSI')
    axes[0].set_ylabel('Price (USD)')

    # RSI
    axes[1].plot(df_plot.index, df_plot['RSI'], color='purple', linewidth=1)
    axes[1].axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought (70)')
    axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
    axes[1].fill_between(df_plot.index, df_plot['RSI'], 70,
                          where=(df_plot['RSI'] >= 70), alpha=0.3, color='red')
    axes[1].fill_between(df_plot.index, df_plot['RSI'], 30,
                          where=(df_plot['RSI'] <= 30), alpha=0.3, color='green')
    axes[1].set_ylabel('RSI')
    axes[1].set_ylim(0, 100)
    axes[1].legend(loc='upper left', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'../data/raw/{ticker}_rsi.png', dpi=150)
    plt.show()

for ticker in TICKERS:
    plot_rsi(ticker, stocks[ticker])

In [ ]:
def plot_macd(ticker, df, start='2022-01-01'):
    df_plot = df[df.index >= start].copy()

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True,
                              gridspec_kw={'height_ratios': [3, 1.5, 1.5]})

    # Price
    axes[0].plot(df_plot.index, df_plot['Adj Close'], color='black', linewidth=1.2)
    axes[0].set_title(f'{ticker} — Price, MACD & Volume')
    axes[0].set_ylabel('Price (USD)')

    # MACD
    axes[1].plot(df_plot.index, df_plot['MACD'],        label='MACD',   color='steelblue', linewidth=1)
    axes[1].plot(df_plot.index, df_plot['MACD_Signal'], label='Signal', color='orange',    linewidth=1)
    colors = ['green' if v >= 0 else 'red' for v in df_plot['MACD_Hist']]
    axes[1].bar(df_plot.index, df_plot['MACD_Hist'], color=colors, alpha=0.5, label='Histogram')
    axes[1].axhline(0, color='gray', linewidth=0.5)
    axes[1].set_ylabel('MACD')
    axes[1].legend(loc='upper left', fontsize=8)

    # Volume
    axes[2].bar(df_plot.index, df_plot['Volume'], color='steelblue', alpha=0.5)
    axes[2].set_ylabel('Volume')

    plt.tight_layout()
    plt.savefig(f'../data/raw/{ticker}_macd.png', dpi=150)
    plt.show()

for ticker in TICKERS:
    plot_macd(ticker, stocks[ticker])

In [ ]:
plt.figure(figsize=(14, 6))

for ticker, df in stocks.items():
    df_plot = df.dropna(subset=['cumulative_return'])
    plt.plot(df_plot.index, df_plot['cumulative_return'] * 100,
             label=ticker, linewidth=1.2)

plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.title('Cumulative Returns Comparison')
plt.xlabel('Date')
plt.ylabel('Cumulative Return (%)')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/cumulative_returns_comparison.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

for ticker, df in stocks.items():
    plt.plot(df.index, df['rolling_vol_20'], label=ticker, linewidth=1)

plt.title('20-Day Rolling Volatility Comparison')
plt.xlabel('Date')
plt.ylabel('Rolling Std of Daily Returns (%)')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/rolling_volatility.png', dpi=150)
plt.show()